In [ ]:
# ruff: noqa: F401
import os
from datetime import datetime
from pathlib import Path

import backend
from agents.mcp import MCPServerStdio, create_static_tool_filter
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from agents import Agent, Runner, trace

load_dotenv(override=True)
console = Console()

### Using Long term memory


In [26]:
memory_path = os.path.abspath("memory/memory.json")
if os.path.isdir(memory_path):
    os.rmdir(memory_path)
os.makedirs(os.path.dirname(memory_path), exist_ok=True)

memory_params = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-memory"],
    "env": {"MEMORY_FILE_PATH": memory_path},
}

async with MCPServerStdio(memory_params, client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

print(*memory_tools, sep="\n")

name='create_entities' title='Create Entities' description='Create multiple new entities in the knowledge graph' input_schema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'type': 'array', 'items': {'type': 'string'}, 'description': 'An array of observation contents associated with the entity'}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities']} execution=ToolExecution(task_support='forbidden') output_schema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'Th

In [27]:
INSTRUCTIONS = """
You are an agent who uses its entiry tools as a persistent memory to store and
recall information about the conversation.
When the request contains durable information, call the memory server's read/ write
tool to store it in memory.json file. Delete the file if it already exists.
Confirm the operation only after the tool call succeeds.
"""

request = """
My name is AG. I am an LLM enginner. I am undergoing a course on Agentic AI. The course includes MCP.
MCP is Model Context Protocol that equips an agent with tools, resources and promptis
from and extenal source.
"""

MODEL = "gpt-5.4-nano"

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="memory_agent", instructions=INSTRUCTIONS, model=MODEL, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    console.print(Markdown(result.final_output))

Saved your details to memory: your name is AG, you’re an LLM engineer taking an Agentic AI course that includes MCP
(Model Context Protocol), which provides tools/resources/prompts from external sources.

In [28]:
# Recall the memory stored

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="memory_agent", instructions=INSTRUCTIONS, model=MODEL, mcp_servers=[mcp_server])
    with trace("Conversation"):
        result = await Runner.run(agent, "My name is AG. What do you know about me?")
    console.print(Markdown(result.final_output))

I know that your name is AG. I also have you down as an LLM engineer who’s taking a course on Agentic AI, including
MCP (Model Context Protocol)—and you noted that MCP equips agents with tools, resources, and prompts from external 
sources.

### Web Search


In [30]:
tavily_params = {
    "command": "npx",
    "args": ["-y", "tavily-mcp@latest"],
    "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")},
}

async with MCPServerStdio(tavily_params, client_session_timeout_seconds=60) as server:
    tools_list = await server.list_tools()

print(*tools_list, sep="\n")

name='tavily_search' title=None description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.' input_schema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date': {'type'

In [31]:
INSTRUCTIONS = """
You are an agent who can search the web and summarize the findings.
"""

TASK = (
    "Please research the latest stock price of Amazon and summarize the results. "
    f"For context, the current date is {datetime.now().date()}."
)

search_only = create_static_tool_filter(allowed_tool_names=["tavily_search"])

async with MCPServerStdio(tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as mcp_server:
    search_agent = Agent(name="search_agent", instructions=INSTRUCTIONS, model=MODEL, mcp_servers=[mcp_server])
    with trace("Search Web"):
        result = await Runner.run(search_agent, TASK)

    console.print(Markdown(result.final_output))

As of 2026-10-04, Amazon.com’s shares (NASDAQ: AMZN) are reported at about $251.52 per share (MarketBeat).         

Summary of the latest available info (from the same source):                                                       

 • AMZN is up ~9.0% compared with $230.82 at the start of 2026.                                                    
 • The cited article also notes a consensus analyst price target of $321.63 (about 27.9% above $251.52).

### Agentic RAG


In [39]:
vectordb_path = Path("memory/qdrant").resolve()
vectordb_path.mkdir(parents=True, exist_ok=True)

vectorstore_params = {
    "command": "uvx",
    "args": ["--python", "3.13", "mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

async with MCPServerStdio(vectorstore_params, client_session_timeout_seconds=60) as server:
    tools = await server.list_tools()

print(*tools, sep="\n")

name='qdrant-find' title=None description='Look up memories in Qdrant. Use this tool when you need to: \n - Find memories by their content \n - Access memories for further analysis \n - Get some personal information about the user' input_schema={'properties': {'query': {'description': 'What to search for', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'type': 'object'} execution=None output_schema=None icons=None annotations=None meta=None
name='qdrant-store' title=None description='Keep the memory for later use, when you are asked to remember something.' input_schema={'properties': {'information': {'description': 'Text to store', 'title': 'Information', 'type': 'string'}, 'metadata': {'anyOf': [{'additionalProperties': True, 'type': 'object'}, {'type': 'null'}], 'default': None, 'description': 'Extra metadata stored along with memorised information. Any json is accepted.', 'title': 'Metadata'}}, 'required': ['information'], 'type': 'object'} execution=None output_schema

In [ ]:
INSTRUCTIONS = """
You are an agent to search topics on the web and build up knowledgebase for later use.
When you learn something worth storing, store it in the knowledge base.
When you are asked a question fiest search the knowledgebase for an answer.
If not found, search from the web and summarize the results and store them on the knowledgebase.
"""

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60) as search_mcp:
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=60) as store_mcp:
        rag_agent = Agent("search_agent", instructions=INSTRUCTIONS, model=MODEL, mcp_servers=[search_mcp, store_mcp])
        with trace("Search Agent"):
            result = await Runner.run(
                rag_agent, input=f"Reasearch the latest news on NVIDIA as on {datetime.now().date()}", max_turns=20
            )

        console.print(Markdown(result.final_output))

As of 2026-10-04, the latest NVIDIA-related items I found online are:                                              

 • NVIDIA DGX Spark (64GB) announced: a desktop AI configuration with 64GB unified memory, starting at $4,999, with
   availability on Oct. 23, 2026 (reported by multiple outlets; positioned as a lower-entry tier while keeping     
   NVIDIA’s DGX OS / AI software stack and GB10 Grace Blackwell platform).                                         
   Sources: https://techaeris.com/2026/10/03/nvidia-unveils-64gb-dgx-spark-at-4999 ,                               
   https://unixgaming.org/nvidia-dgx-spark-64gb-launch-2026 ,                                                      
   https://thepcenthusiast.com/nvidia-dgx-spark-64gb-price-release-date                                            
 • NVIDIA “Open Agent Safety Platform” (late Sep 2026; still among the newest items discussed): described as a     
   full-stack approach to govern/secure autonomous AI agents, combining OpenShell (runtime governance) and Sentry  
   (monitoring + isolation/quarantine, using BlueField-4 / DPU-based design).                                      
   Source encountered: https://ai-xblog.com/nvidia-open-agent-safety-platform                                      

If you want, tell me whether you mean NVIDIA corporate/newsroom, GPU/product launches, or stock/earnings/finance,  
and I’ll narrow the next pass to that category.

In [41]:
# Search again with the same question that is stored in the knowledgebase
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as server:
    agent = Agent("reasearcher", instructions=INSTRUCTIONS, model=MODEL, mcp_servers=[server])
    with trace("Search RAG"):
        result = await Runner.run(agent, "Based on your knowledgebase find and summarize the latest news on NVIDIA")
    console.print(Markdown(result.final_output))

From my knowledgebase (as of 2026-10-04), the latest NVIDIA items captured are:                                    

 • Oct 2, 2026 — DGX Spark 64GB workstation announced: NVIDIA reportedly introduced a DGX Spark desktop AI         
   workstation featuring 64GB unified memory, with MSRP ~$4,999, and availability starting Oct 23, 2026. Coverage  
   notes it includes the Grace Blackwell (GB10) superchip and NVIDIA’s AI software stack.                          
 • Late Sep 2026 — Open Agent Safety Platform: NVIDIA reportedly launched an “Open Agent Safety Platform” aimed at 
   securing autonomous AI agents, combining an agent runtime/governance layer (OpenShell) with                     
   monitoring/quarantine/isolation (Sentry)—with references to BlueField-4-based monitoring.                       

If you want, I can also refresh this by doing a fresh web search for today’s headlines and cross-checking sources.

### Using Market Reasearch Data MCP Server (Massive)


In [ ]:
massive_api_key = os.getenv("MASSIVE_API_KEY")

if massive_api_key:
    market_params = {
        "command": "uvx",
        "args": [
            "--with",
            "mcp>=1.27.0,<2",
            "--from",
            "git+https://github.com/massive-com/mcp_massive@v0.10.0",
            "mcp_massive",
        ],
        "env": {"MASSIVE_API_KEY": massive_api_key},
    }
else:
    market_params = {
        "command": "uv",
        "args": ["run", "-m", "backend.market_server"],
    }


async with MCPServerStdio(market_params, client_session_timeout_seconds=60) as market_mcp:
    tools_list = await market_mcp.list_tools()

print(*tools_list, sep="\n")

name='search_endpoints' title=None description='Search for market data API endpoints and built-in finance functions by natural language query. Use this FIRST to find the right endpoint before calling call_api. Covers stocks, options, forex, crypto, futures, indices, ETFs, and economic data. Pass market to pin results to a specific asset class when you already know it; omit it and the server will infer from the query. Use detail="more" to see query parameter docs needed for building call_api requests.' input_schema={'additionalProperties': False, 'properties': {'query': {'description': 'Natural language search query for API endpoints', 'minLength': 1, 'title': 'Query', 'type': 'string'}, 'scope': {'anyOf': [{'enum': ['all', 'endpoints', 'functions'], 'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Search scope: "endpoints" for API only, "functions" for local functions only, or "all"/omit for both', 'title': 'Scope'}, 'max_results': {'anyOf': [{'maximum': 25, 'mini

In [47]:
INSTRUCTIONS = """
You are an agent to answer questions on stock markets
"""

async with MCPServerStdio(market_params, client_session_timeout_seconds=60) as market_mcp:
    market_agent = Agent("market_agent", instructions=INSTRUCTIONS, model=MODEL, mcp_servers=[market_mcp])
    with trace("Market Reasearch"):
        result = await Runner.run(market_agent, "What is the most recent price that Apple (AAPL) is traded at?")
    console.print(Markdown(result.final_output))

I can’t retrieve AAPL’s most recent traded price from the market data API with the current access level (the       
latest-trade endpoint is blocked with a 403 “NOT_AUTHORIZED”).

### Stock information using Yahoo Finance MCP


In [63]:
yf_params = {
    "command": "uvx",
    "args": ["yfinance-mcp-server"],
}

INSTRUCTIONS = """
You answer questions about US stocks using the yfinance tools.
When quoting a price, say whether the market is open or closed and give the
timestamp. If the market is closed, call it the last close, not the current price.
"""

async with MCPServerStdio(
    yf_params,
    client_session_timeout_seconds=60,
    cache_tools_list=True,
    tool_filter=create_static_tool_filter(allowed_tool_names=["get_stock_info"]),
) as yf_mcp:
    yf_agent = Agent("yf_agent", instructions=INSTRUCTIONS, model="gpt-5.4-nano", mcp_servers=[yf_mcp])
    result = await Runner.run(yf_agent, "What is the most recent price that Apple (AAPL) is traded at?")

    console.print(Markdown(result.final_output))


Apple (AAPL) was last traded at $333.69 (market closed). Timestamp: 2026-10-04 20:01:46 UTC (regular session).

In [62]:
async with MCPServerStdio(yf_params, client_session_timeout_seconds=60) as yf_mcp:
    tools = await yf_mcp.list_tools()
    print(len(tools))
    print(*tools, sep="\n")

23
name='get_stock_info' title='Get Stock Info' description='Get comprehensive company information for a stock ticker.' input_schema={'type': 'object', 'additionalProperties': False, 'properties': {'symbol': {'type': 'string', 'description': "Stock ticker symbol (e.g., 'AAPL', 'MSFT')"}}, 'required': ['symbol']} execution=None output_schema={'additionalProperties': True, 'type': 'object'} icons=None annotations=None meta={'fastmcp': {'tags': []}}
name='get_stock_history' title='Get Stock History' description='Get historical stock price data.' input_schema={'type': 'object', 'additionalProperties': False, 'properties': {'symbol': {'type': 'string', 'description': 'Stock ticker symbol'}, 'period': {'default': '1mo', 'type': 'string', 'description': 'Time period (1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max)'}, 'interval': {'default': '1d', 'type': 'string', 'description': 'Data interval (1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 1d, 5d, 1wk, 1mo, 3mo)'}, 'start': {'anyOf': [{'type': 'strin